In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Make sure this matches exactly what you trained!
# BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-14B-Instruct" 
BASE_MODEL_NAME = "LLM4Binary/llm4decompile-9b-v2" 
ADAPTER_DIR = "./final_lora_model"

print("Loading base model in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)


base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Loading tokenizer and adapter...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

In [ ]:

# Format the prompt exactly how it was trained
prompt = """<|im_start|>system
You are an expert programming assistant. You must always think step-by-step inside <scratchpad> tags before providing your final answer.<|im_end|>
<|im_start|>user
Write a simple Python function to add two numbers.<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Generating response...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512, # Adjust based on how long your bytecode gets
        temperature=0.2,    # Keep low for code generation
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\n=== MODEL OUTPUT ===\n")
print(response)